# Katalog danych i wybór uczestnika

Jak sprawdzić przydział rodzin oraz powiązane pliki przed wczytaniem dużych źródeł? Notebook korzysta z lokalnych plików przez bibliotekę `aminor_msa`; nie uruchamia pobierania ani nie rozpakowuje całego ZIP-a. Każdy notebook można uruchomić osobno przez **Restart Kernel → Run All**.

## Introduction

Zmień wybór w `settings.toml` albo w jawnie nazwanych zmiennych przy wywołaniu. Kontrolki działają w uruchomionym kernelu; po zmianie rodziny w kontrolce uruchom ponownie komórki, które odczytują `browser.selected_family`.

## Assumptions

- Środowisko: `uv sync --extra notebook`; w edytorze wybierz `.venv/bin/python`.
- Dane są już w `data/`. Jeśli ich brakuje, użyj z katalogu projektu `uv run python -m assets.scripts.download`; szczegóły w [acquisition.md](../../../docs/acquisition.md).
- To kontrola formatów i jawnych kluczy źródłowych. Nie powstaje mapowanie reszt struktury do kolumn MSA ani zbiór treningowy.
- Przykład wybiera Antoninę Olejnik, rodzinę Fluoride `RF01734` i strukturę `3vrs`. Inne rodziny mogą nie mieć struktur lub raportów DSSR.

## Notation

`participant_id` to klucz przydziału; `rfam_acc` identyfikuje rodzinę, `pdb_id` strukturę. Jeden PDB może należeć do wielu rodzin; jeden wiersz katalogu opisuje konkretny łańcuch i zakres.

[Reader: odczyt, klucze i przygotowanie do treningu](../../../docs/reader.md).

In [ ]:
from pathlib import Path
from IPython.display import display
import pandas as pd

from aminor_msa import DataStore
from aminor_msa.utils.config import read_toml
from aminor_msa.utils.paths import find_project_root
from aminor_msa.visualization import ViewStyle

ROOT = find_project_root()
NOTEBOOK_DIR = ROOT / "assets/analysis/base_functionality/26_10_06"
SETTINGS_PATH = NOTEBOOK_DIR / "settings.toml"
settings = read_toml(SETTINGS_PATH)
selection = settings["selection"]
limits = settings["display"]
style = ViewStyle(**{key: limits[key] for key in ("max_sequences", "max_columns", "font_size", "width", "height")})
store = DataStore(ROOT / "data")
display({"data_root": str(store.root), "selection": selection, "initial_state": store.load_state()})

## Participants and families

### Methodology

**Theoretical methodology.** Przydział uczestnika pochodzi wyłącznie z `data/config/families.csv`, nie z nazwy pliku mmCIF. Filtr uczestnika jest niewrażliwym na wielkość liter podciągiem; filtr RNA dopasowuje cały token oddzielony średnikiem.

**Implementation methodology.** `participants_table` i `families_table` czytają mały manifest. Nie parsują MSA, współrzędnych ani treści DSSR.

**Figure description.** Tabela uczestników podaje liczbę unikalnych rodzin; następna tabela zachowuje wszystkie kolumny przydziału.

In [ ]:
from aminor_msa.visualization import participants_table, families_table

participant = selection["participant"]
rna_type = 'riboswitch'  # Try "riboswitch" or "ribozyme".
entry_type = None  # Exact manifest entry type, if filtering is needed.
participants = participants_table(store)
families = families_table(store, participant=participant, rna_type=rna_type, entry_type=entry_type)
display(participants)
display(families)
assert not families.empty, "No families match this selection; change the filters."
assert set(families["rfam_acc"]) == set(store.family_ids(participant=participant, rna_type=rna_type, entry_type=entry_type))

### Remarks

`rna_type="Gene"` i `rna_type="riboswitch"` mogą wybierać tę samą rodzinę: pole zawiera kilka tokenów. Pusta tabela jest poprawnym wynikiem filtra.

### Notes

## Interactive selection

### Methodology

**Theoretical methodology.** Relacja rodzina → PDB jest wielowartościowa. Zachowujemy wszystkie wiersze łańcuchów i zakresów, a listę plików PDB deduplikujemy po identyfikatorze.

**Implementation methodology.** `FamilyBrowser` aktualizuje tabele po zmianie uczestnika, typu lub rodziny. Odczytuje tylko manifest, `Rfam.pdb`, metadane plików i nazwy członków ZIP-a.

**Figure description.** Trzy listy wyboru oraz tabele rodzin, mapowań katalogowych i dostępnych struktur. Brak wariantów oznacza brak raportu w archiwum, nie ujemną etykietę biologiczną.

In [ ]:
from aminor_msa.visualization import FamilyBrowser

participant = selection["participant"]
accession = selection["family"]
browser = FamilyBrowser(store, participant=participant, accession=accession)
display(browser.widget)

In [ ]:
# Re-run this cell after changing the family in the widget.
from aminor_msa.visualization import mappings_table, structures_table

family = browser.selected_family
assert family is not None, "The selected filters contain no family."
mappings = mappings_table(family)
structures = structures_table(family)
display({"selected_family": family.accession, "path": str(family.path)})
display(mappings)
display(structures)
display(store.load_state())
assert not store.load_state()["cached_objects"]

### Remarks

`pdb_start/end` to numeracja autora, `cm_start/end` to pozycje modelu Infernal. Żadne z nich nie jest automatycznie numerem kolumny MSA. Sufiks `.out1` identyfikuje raport Baulina; nie wybiera modelu 1 mmCIF.

### Notes

## Reading without exports

Tabele są wyliczane w pamięci z aktualnie wybranych źródeł. Notebook nie zapisuje CSV, metadanych wykonania ani cache na dysku. Ponowne uruchomienie odtwarza podglądy.

### Remarks

Odczyt nazw ZIP-a nie dekompresuje raportów. Samo wyświetlenie uchwytu `family` również nie ładuje jego MSA.

### Notes

## Results / Summary

### LLM

Liczba wybranych rodzin wynika z bieżących filtrów. Kontrolki pokazują ich powiązania katalogowe i dostępność lokalnych plików; nie odczytują ciężkich payloadów. Tabele pozostają w pamięci bieżącej sesji.

### Person